## Optimizing & pruning taking a while because of lots of small files

In [0]:
describe detail alexn.default.web_sales_small_files

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,1cfcb720-76f5-4ef5-8830-155692891cc5,alexn.default.web_sales_small_files,null,abfss://unity@unitydemo.dfs.core.windows.net/b86c6879-8c55-4e70-a585-18d16a4fa6e9/tables/f1e12bea-21ca-4692-8027-834d6ab2d136,2026-06-09T13:04:45.214Z,2026-06-09T13:11:05.000Z,List(),List(),100000,1579436193,"Map(delta.enableDeletionVectors -> true, delta.parquet.compression.codec -> zstd)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numDeletionVectors -> 0, numRowsDeletedByDeletionVectors -> 0)",false


In [0]:
select ws_quantity, count(*) from alexn.default.web_sales_small_files group by 1

ws_quantity,count(*)
94,71662
29,71814
88,72218
56,72004
null,1769
24,72508
99,71711
39,71725
38,71798
15,72150


In [0]:
optimize alexn.default.web_sales_small_files

path,metrics
abfss://unity@unitydemo.dfs.core.windows.net/b86c6879-8c55-4e70-a585-18d16a4fa6e9/tables/f1e12bea-21ca-4692-8027-834d6ab2d136,"List(24, 100000, List(10249839, 19731722, 1.8915952208333332E7, 24, 453982853), List(15507, 16119, 15794.36193, 100000, 1579436193), 0, null, null, 0, 1, 100000, 0, true, 0, 0, 1781010693217, 1781010805576, 232, 24, null, List(0, 0), null, 34, 32, 1010907, 0, null, null)"


In [0]:
create table alexn.default.web_sales_small_files2 clone alexn.default.web_sales_small_files

In [0]:
describe detail alexn.default.web_sales_small_files2

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,c07daa97-dc81-4190-a777-44772c7aa741,alexn.default.web_sales_small_files2,null,abfss://unity@unitydemo.dfs.core.windows.net/b86c6879-8c55-4e70-a585-18d16a4fa6e9/tables/fbdf883b-fd54-4f85-bf00-5d64868df6db,2026-06-09T13:04:45.214Z,2026-06-09T13:21:06.000Z,List(),List(),24,453982853,"Map(delta.enableDeletionVectors -> true, delta.parquet.compression.codec -> zstd)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numDeletionVectors -> 0, numRowsDeletedByDeletionVectors -> 0)",false


In [0]:
select ws_quantity, count(*) from alexn.default.web_sales_small_files2 group by 1

ws_quantity,count(*)
11,72418
38,71798
46,72056
68,72635
null,1769
15,72150
97,72154
33,71664
24,72508
76,71640


## Scan operator -- no pruning

In [0]:
%python

result = spark.sql("select ws_quantity from alexn.default.web_sales where ws_quantity > 50")
result.write.mode("overwrite").saveAsTable("alexn.default.web_sales_quantities")

In [0]:
alter table alexn.default.web_sales cluster by (ws_quantity)

In [0]:
optimize alexn.default.web_sales full

path,metrics
abfss://unity@unitydemo.dfs.core.windows.net/b86c6879-8c55-4e70-a585-18d16a4fa6e9/tables/8921748c-15db-4b76-a80d-5c9420a44724,"List(99, 80, List(55468907, 111528728, 5.983838597979798E7, 99, 5924000212), List(42633031, 78815569, 6.04777288875E7, 80, 4838218311), 0, null, null, 0, 1, 80, 0, false, 0, 0, 1780955071728, 1780955173207, 8, 1, null, List(0, 0), null, 34, 32, 287314, 0, List(4838218311, false, false, true, 0.9892761587511028, List(0.9892761587511028), 1.0, null, 0, 0, 0, 0, 80, 4838218311, 4838218311, null, log, 16777216, 67108864, 4, 0, 0, List(94), 1, 0, 0, 0, 0, 1, 1, 1, 1, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4838218311, 4838218311, List(318, 2319, 1105, 669, 1273, 0), 2, 1, 5, default, false, 0, null, false, 0, 0, 0, null, null, null), null)"
abfss://unity@unitydemo.dfs.core.windows.net/b86c6879-8c55-4e70-a585-18d16a4fa6e9/tables/8921748c-15db-4b76-a80d-5c9420a44724,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 99, 0, false, 0, 0, 1780955173264, 1780955177062, 8, 0, null, List(0, 0), null, 34, 32, 0, 0, List(5924000212, false, false, true, 0.9892761587511028, List(0.9892761587511028), 1.0, null, 0, 0, 0, 0, 0, 0, 0, null, log, 16777216, 67108864, 4, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(142, 18, 639, 0, 0, 0), 2, 2, 5, default, false, 0, null, false, 0, 0, 0, null, null, null), null)"
abfss://unity@unitydemo.dfs.core.windows.net/b86c6879-8c55-4e70-a585-18d16a4fa6e9/tables/8921748c-15db-4b76-a80d-5c9420a44724,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 99, 0, true, 0, 0, 1780955177108, 1780955180101, 8, 0, null, List(0, 0), null, 34, 32, 0, 0, List(5924000212, false, false, false, 0.9892761587511028, List(0.9892761587511028), 1.0, post-optimize-compaction, 0, 0, 0, 0, 0, 0, 0, null, null, 33554432, 67108864, 0, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(0, 0, 569, 0, 0, 0), 15, 1, 1, null, false, 0, null, false, 0, 0, 0, null, null, null), null)"
abfss://unity@unitydemo.dfs.core.windows.net/b86c6879-8c55-4e70-a585-18d16a4fa6e9/tables/8921748c-15db-4b76-a80d-5c9420a44724,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 99, 0, true, 0, 0, 1780955180886, 1780955181335, 8, 0, null, List(0, 0), SNAPPY, 34, 32, 0, 0, null, null)"


In [0]:
%python

result = spark.sql("select ws_quantity from alexn.default.web_sales where ws_quantity > 50")
result.write.mode("overwrite").saveAsTable("alexn.default.web_sales_quantities")

Only 12MB read after it was able to prune the files, speeding up the query.

## Shuffle / sort operators -- lots of spill

In [0]:
SELECT
  COUNT(*)           AS row_count,
  MAX(running_total) AS max_running_total
FROM (
  SELECT
    SUM(id) OVER (
      ORDER BY notes                                   -- no PARTITION BY => all rows in ONE partition/task
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW -- true running sum => the sort cannot be skipped
    ) AS running_total
  FROM (
    SELECT
      id,
      repeat(concat('payload-', CAST(id AS string), '-'), 20) AS notes  -- wide (~240 char) sort key
    FROM range(0, 80000000)                                             -- 80M rows
  )
);

row_count,max_running_total
80000000,3199999960000000


In [0]:
SELECT
  COUNT(*) AS row_count,
  SUM(id)  AS max_running_total
FROM range(0, 80000000);

row_count,max_running_total
80000000,3199999960000000


Refactored query removed the spill and it ran faster.